# AMS 580 Group 1 Project Notebook

## Notebook Roadmap

1. Setup and data loading
2. Exploratory data quality checks
3. Data cleaning and feature matrix preparation
4. Classical model selection and tuning
5. Transformer-based model training and comparison
6. Review notes and red flags

This notebook keeps the classical ML workflow and transformer workflow as separate modeling tracks. Run cells from top to bottom within each track so reused variable names are refreshed in the intended order.


## 1. Setup

Install runtime dependencies, import shared libraries, and load the project data.


### 1.1 Dependencies

Install notebook-only dependencies needed by the modeling workflow.


In [ ]:
#pip install optuna

### 1.2 Imports

Load the common Python, scikit-learn, imbalanced-learn, and optimization libraries used by the classical modeling track.


In [ ]:
import pandas as pd
import numpy as np
import optuna
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, balanced_accuracy_score, roc_auc_score, make_scorer, accuracy_score, recall_score
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate, RandomizedSearchCV, GridSearchCV

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.3 Data Loading

Load the provided training and testing files. The active paths assume a Colab runtime; the commented lines show the local workspace alternative.


In [ ]:
# colab
train = pd.read_csv("./training_data.csv")
test = pd.read_csv("./testing_data.csv")

# local workspace
# train = pd.read_csv("./data/training_data.csv")
# test = pd.read_csv("./data/testing_data.csv")

## 2. Exploratory Data Quality Checks

Inspect schema, missing values, encoded `unknown` categories, class balance, numeric summaries, and correlations among economic indicators before modeling.


In [ ]:
print(train.dtypes)

print("\n=== MISSING / UNKNOWN COUNTS ===")
for col in train.columns:
    unknowns = (train[col] == 'unknown').sum()
    nulls = train[col].isnull().sum()
    if unknowns > 0 or nulls > 0:
        print(f"  {col}: {nulls} nulls, {unknowns} unknowns")

age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

=== MISSING / UNKNOWN COUNTS ===
  job: 0 nulls, 269 unknowns
  marital: 0 nulls, 63 unknowns
  education: 0 nulls, 1398 unknowns
  default: 0 nulls, 6916 unknowns
  housing: 0 nulls, 804 unknowns
  loan: 0 nulls, 804 unknowns


In [ ]:
# Check if housing and loan has overlapping "Unknown" values
overlap = ((train['housing'] == 'unknown') & (train['loan'] == 'unknown')).sum()
print(f"Numbers of Rows where BOTH housing and loan are unknown: {overlap}")
print(f"Housing unknowns: {(train['housing'] == 'unknown').sum()}")
print(f"Loan unknowns: {(train['loan'] == 'unknown').sum()}")


print("=== TARGET DISTRIBUTION ===") #y is marketing success (Yes/No)
print(train['y'].value_counts())
print(train['y'].value_counts(normalize=True).round(3))


print("=== NUMERIC SUMMARY ===")
num_cols = train.select_dtypes(include='number').columns.tolist()
print(train[num_cols].describe().round(2))

Numbers of Rows where BOTH housing and loan are unknown: 804
Housing unknowns: 804
Loan unknowns: 804
=== TARGET DISTRIBUTION ===
y
no     29239
yes     3712
Name: count, dtype: int64
y
no     0.887
yes    0.113
Name: proportion, dtype: float64
=== NUMERIC SUMMARY ===
            age  duration  campaign     pdays  previous  emp.var.rate  \
count  32951.00  32951.00  32951.00  32951.00  32951.00      32951.00   
mean      40.04    257.17      2.55    961.84      0.17          0.08   
std       10.43    258.69      2.76    188.46      0.49          1.57   
min       17.00      0.00      1.00      0.00      0.00         -3.40   
25%       32.00    102.00      1.00    999.00      0.00         -1.80   
50%       38.00    179.00      2.00    999.00      0.00          1.10   
75%       47.00    318.00      3.00    999.00      0.00          1.40   
max       98.00   4918.00     56.00    999.00      6.00          1.40   

       cons.price.idx  cons.conf.idx  euribor3m  nr.employed  
count     

In [ ]:
econ_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
print(train[econ_cols].corr().round(2))

                emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  \
emp.var.rate            1.00            0.77           0.20       0.97   
cons.price.idx          0.77            1.00           0.06       0.69   
cons.conf.idx           0.20            0.06           1.00       0.28   
euribor3m               0.97            0.69           0.28       1.00   
nr.employed             0.91            0.52           0.10       0.95   

                nr.employed  
emp.var.rate           0.91  
cons.price.idx         0.52  
cons.conf.idx          0.10  
euribor3m              0.95  
nr.employed            1.00  


## 3. Data Cleaning and Feature Matrix Preparation

Remove the known leakage feature (`duration`), engineer campaign/contact indicators, cap campaign outliers, encode the binary target, one-hot encode categorical variables, and align train/test feature columns.


In [ ]:
train_df = train.copy()
test_df = test.copy()

# Drop 'duration' column — target leakage because duration is known after the call ends and by that time the outcome of y is already captured.
# Including "duration" column will inflate the prediction accuracy of y since longer duration will most likely mean a positive outcome.
# Since the goal here is to build a robust model, removing 'duration' is an important step.
train_df.drop(columns=['duration'], inplace=True)
test_df.drop(columns=['duration'], inplace=True)

# drop highly correlated economic indicators
# emp.var.rate (r=0.97 with euribor3m) and nr.employed (r=0.95 with euribor3m) and can complicate model learning
# We retain euribor3m as it is the most granular (daily) economic indicator.
#train_df.drop(columns=['emp.var.rate', 'nr.employed'], inplace=True)
#test_df.drop(columns=['emp.var.rate', 'nr.employed'], inplace=True)

# Engineer 'was_contacted_before' binary flag. pdays=999 means the client was NEVER contacted in a previous campaign.
# This binary flag explicitly captures that distinction, which is highly
# predictive — previously contacted clients behave very differently.
train_df['was_contacted_before'] = (train_df['pdays'] != 999).astype(int)
test_df['was_contacted_before']  = (test_df['pdays'] != 999).astype(int)

# Cap 'campaign' at 95th percentile. Campaign has extreme outliers (max=56, mean=2.5).
# Outliers can distort distance-based models. Capping at the 95th percentile reduces their influence while preserving the distribution shape.
cap_value = train_df['campaign'].quantile(0.95)
train_df['campaign'] = train_df['campaign'].clip(upper=cap_value)
test_df['campaign']  = test_df['campaign'].clip(upper=cap_value)
print(f"Campaign capped at: {cap_value}")

# Encode target variable. Convert y from string ("yes"/"no") to binary integer (1/0).
train_df['y'] = (train_df['y'] == 'yes').astype(int)
test_df['y']  = (test_df['y'] == 'yes').astype(int)

#Train-test split
X_train = train_df.drop(columns=['y'])
y_train = train_df['y']
X_test  = test_df.drop(columns=['y'])
y_test  = test_df['y']

# One-hot encode categorical variables. We one-hot encode all categoricals including "unknown" as its own category since "unknown" may itself carry predictive signal
# drop_first=False keeps all categories for interpretability.
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=False)
X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=False)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
print(f"Features after encoding — Train: {X_train.shape}, Test: {X_test.shape}")

# Scaling and SMOTE done BEFORE cross-validation causes data leakage because information from the full training set leaks into each validation fold.
# Instead, scaling and SMOTE must happen INSIDE the cross val pipeline so they are fit only on each fold's training split.
X_train_ready = X_train.copy()
X_test_ready  = X_test.copy()

print(f"\nClass distribution before SMOTE: {y_train.value_counts().to_dict()}")
print(f"Final shapes — X_train: {X_train_ready.shape}, X_test: {X_test_ready.shape}")

Campaign capped at: 7.0
Categorical columns to encode: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Features after encoding — Train: (32951, 63), Test: (8237, 63)

Class distribution before SMOTE: {0: 29239, 1: 3712}
Final shapes — X_train: (32951, 63), X_test: (8237, 63)


/var/folders/4l/f5ccm6754c3gcqsbf4bhmz2h0000gn/T/ipykernel_23832/31991407.py:41: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include='object').columns.tolist()


## 4. Classical Model Selection and Tuning

Compare baseline classical models under class imbalance using balanced accuracy, sensitivity, specificity, and AUC-style summaries. SMOTE and scaling are placed inside pipelines so fold-specific preprocessing happens during cross-validation.


### 4.1 Cross-Validated Model Comparison

Evaluate logistic regression, random forest, XGBoost, LightGBM, and KNN with class-balancing strategies under 5-fold stratified cross-validation.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import subprocess

# Scoring metrics
scoring_bal = {
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'accuracy':          make_scorer(accuracy_score),
    'sensitivity':       make_scorer(recall_score, pos_label=1),
    'specificity':       make_scorer(recall_score, pos_label=0),
    'auc':               make_scorer(roc_auc_score),
}
cv_bal = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 5-Fold Stratified Cross Validation. Stratified ensures each fold maintains the same class ratio.
# We use the original training data, with scaling/SMOTE happening inside each fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# For Logistic Regression and KNN: StandardScaler is needed because these models are sensitive to feature scale.
# SMOTE is applied after scaling so nearest-neighbor synthesis happens in a properly scaled feature space.

# For RF, KNN, and XGBoost: SMOTE is applied inside each fold.
# For LogReg, we opted to use the native class imbalance argument
# For XGBoost, we used both SMOTE and the native class imbalance argument

def gpu_available():
    try:
        subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

USE_GPU = gpu_available()
print("GPU available:", USE_GPU)
xgb_device   = 'cuda' if USE_GPU else 'cpu'
lgb_device   = 'gpu'  if USE_GPU else 'cpu'
import numpy as np
neg, pos = np.bincount(y_train)
scale_pos = neg / pos

models_bal = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')),
    ]),
    'Random Forest': Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1
        )),
    ]),
    'XGBoost': Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', xgb.XGBClassifier(
            eval_metric='logloss', random_state=42, n_jobs=-1, device=xgb_device, scale_pos_weight=scale_pos
        )),
    ]),
    'LightGBM': Pipeline([
        ('model', lgb.LGBMClassifier(
            random_state=42, n_jobs=-1, verbose=-1,
            device=lgb_device, class_weight='balanced'
        )),
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ]),
}

# Run CV for all models
results_bal = {}

for name, model in models_bal.items():
    print(f"Training {name} (balanced)...")
    cv_res = cross_validate(model, X_train_ready, y_train, cv=cv_bal, scoring=scoring_bal, n_jobs=-1)
    results_bal[name] = {
        'Balanced Acc': cv_res['test_balanced_accuracy'].mean().round(4),
        'Accuracy':     cv_res['test_accuracy'].mean().round(4),
        'Sensitivity':  cv_res['test_sensitivity'].mean().round(4),
        'Specificity':  cv_res['test_specificity'].mean().round(4),
        'AUC':          cv_res['test_auc'].mean().round(4),
    }
    print("  ✓ Done")

# Summary table
results_df = pd.DataFrame(results_bal).T
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\n=== 5-FOLD CV RESULTS ===")
print(results_df.to_string())

GPU available: False
Training Logistic Regression (balanced)...
  ✓ Done
Training Random Forest (balanced)...
  ✓ Done
Training XGBoost (balanced)...
  ✓ Done
Training LightGBM (balanced)...
  ✓ Done
Training KNN (balanced)...
  ✓ Done

=== 5-FOLD CV RESULTS ===
                     Balanced Acc  Accuracy  Sensitivity  Specificity     AUC
Random Forest              0.6595    0.8884       0.3640       0.9550  0.6595
LightGBM                   0.7566    0.8462       0.6409       0.8723  0.7566
Logistic Regression        0.7465    0.8283       0.6409       0.8521  0.7465
XGBoost                    0.7325    0.8190       0.6210       0.8441  0.7325
KNN                        0.6886    0.7588       0.5981       0.7792  0.6886


### 4.2 Hyperparameter Search

Tune LightGBM, random forest, and XGBoost with balanced-accuracy scoring, then fit the best estimators on the full training set.


In [ ]:
# Section 2: Hyperparameter tuning with SMOTE + balanced_accuracy scoring

tune_cv_bal = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── LightGBM (is_unbalance) ─────────────────────────────────────────
lgb_pipeline_bal = Pipeline([
    ('model', lgb.LGBMClassifier(
        random_state=42, n_jobs=-1, verbose=-1,
        device=lgb_device, is_unbalance=True # native unbalanced data handling
    )),
])

lgb_param_grid_bal = {
    'model__n_estimators':     [100, 200, 300],
    'model__learning_rate':    [0.05, 0.1],
    'model__num_leaves':       [31, 63, 127],
    'model__max_depth':        [-1, 10, 20],
    'model__subsample':        [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
}

lgb_search_bal = RandomizedSearchCV(
    lgb_pipeline_bal, lgb_param_grid_bal,
    n_iter=16, cv=tune_cv_bal, scoring='balanced_accuracy',
    random_state=42, n_jobs=1, verbose=1
)
print("Tuning LightGBM (balanced)...")
lgb_search_bal.fit(X_train_ready, y_train)
print(f"Best LGB Params:       {lgb_search_bal.best_params_}")
print(f"Best LGB Balanced Acc: {lgb_search_bal.best_score_:.4f}")

# ── Random Forest (SMOTE + balanced_subsample) ───────────────────────────────
rf_pipeline_bal = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(
        random_state=42, n_jobs=-1
    )),
])

rf_param_grid_bal = {
    'model__n_estimators':      [100, 200, 300],
    'model__max_depth':         [10, 20, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf':  [1, 2],
    'model__max_features':      ['sqrt', 'log2'],
}

rf_search_bal = GridSearchCV(
    rf_pipeline_bal, rf_param_grid_bal,
    cv=tune_cv_bal, scoring='balanced_accuracy',
    n_jobs=1, verbose=1
)
print("\nTuning Random Forest (balanced)...")
rf_search_bal.fit(X_train_ready, y_train)
print(f"Best RF Params:        {rf_search_bal.best_params_}")
print(f"Best RF Balanced Acc:  {rf_search_bal.best_score_:.4f}")

# ── XGBoost (SMOTE + scale_pos_weight) ──────────────────────────────────────
xgb_pipeline_bal = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', xgb.XGBClassifier(
        eval_metric='logloss', tree_method='hist',
        device=xgb_device, random_state=42, n_jobs=-1, scale_pos_weight=scale_pos
    )),
])

xgb_param_grid_bal = {
    'model__n_estimators':     [100, 200, 300],
    'model__max_depth':        [3, 4, 5],
    'model__learning_rate':    [0.05, 0.1],
    'model__subsample':        [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
}

xgb_search_bal = RandomizedSearchCV(
    xgb_pipeline_bal, xgb_param_grid_bal,
    n_iter=50, cv=tune_cv_bal, scoring='balanced_accuracy',
    random_state=42, n_jobs=1, verbose=1
)
print("\nTuning XGBoost (balanced)...")
xgb_search_bal.fit(X_train_ready, y_train)
print(f"Best XGB Params:       {xgb_search_bal.best_params_}")
print(f"Best XGB Balanced Acc: {xgb_search_bal.best_score_:.4f}")

# Summary
tuned_bal_summary = pd.DataFrame({
    'Model': ['Tuned LightGBM (Bal)', 'Tuned Random Forest (Bal)', 'Tuned XGBoost (Bal)'],
    'Best CV Balanced Acc': [
        round(lgb_search_bal.best_score_, 4),
        round(rf_search_bal.best_score_, 4),
        round(xgb_search_bal.best_score_, 4),
    ]
}).sort_values('Best CV Balanced Acc', ascending=False)
print("\n=== TUNED MODEL BALANCED CV RESULTS ===")
print(tuned_bal_summary.to_string(index=False))

# Fit on full training data
best_lgb_bal = lgb_search_bal.best_estimator_
best_rf_bal  = rf_search_bal.best_estimator_
best_xgb_bal = xgb_search_bal.best_estimator_

best_lgb_bal.fit(X_train_ready, y_train)
best_rf_bal.fit(X_train_ready, y_train)
best_xgb_bal.fit(X_train_ready, y_train)

lgb_pred_bal = best_lgb_bal.predict(X_test_ready)
rf_pred_bal  = best_rf_bal.predict(X_test_ready)
xgb_pred_bal = best_xgb_bal.predict(X_test_ready)
print("\nBest balanced estimators fitted on full training data.")


Tuning LightGBM (balanced)...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best LGB Params:       {'model__subsample': 1.0, 'model__num_leaves': 31, 'model__n_estimators': 100, 'model__max_depth': 10, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}
Best LGB Balanced Acc: 0.7566

Tuning Random Forest (balanced)...
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best RF Params:        {'model__max_depth': 10, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 300}
Best RF Balanced Acc:  0.7428

Tuning XGBoost (balanced)...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best XGB Params:       {'model__subsample': 0.8, 'model__n_estimators': 300, 'model__max_depth': 5, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}
Best XGB Balanced Acc: 0.7443

=== TUNED MODEL BALANCED CV RESULTS ===
                    Model  Best CV Balanced Acc
     Tuned LightGBM (Bal

### 4.3 Optuna Search: LightGBM

Run a wider Optuna search for LightGBM and evaluate the selected pipeline on the test set.


In [ ]:
# Optuna for LightGBM — Section 2 (balanced_accuracy)

def lgb_objective_bal(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 127),
        'max_depth':         trial.suggest_int('max_depth', 5, 20),
        'subsample':         trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
    }
    pipe = Pipeline([
        ('model', lgb.LGBMClassifier(
            **params,
            random_state=42, n_jobs=1, verbose=-1, device=lgb_device, is_unbalance=True
        )),
    ])
    scores = cross_val_score(
        pipe, X_train_ready, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='balanced_accuracy', n_jobs=1
    )
    return scores.mean()

lgb_study_bal = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
lgb_study_bal.optimize(lgb_objective_bal, n_trials=75)

print(f"Best LGB Optuna (balanced) params: {lgb_study_bal.best_params}")
print(f"Best LGB Optuna balanced accuracy: {lgb_study_bal.best_value:.4f}")

best_lgb_optuna_bal = Pipeline([
    ('model', lgb.LGBMClassifier(
        **lgb_study_bal.best_params,
        random_state=42, n_jobs=-1, verbose=-1, device=lgb_device, is_unbalance=True
    )),
])
best_lgb_optuna_bal.fit(X_train_ready, y_train)
lgb_optuna_pred_bal = best_lgb_optuna_bal.predict(X_test_ready)

lgb_optuna_bal_results = pd.DataFrame([{
    'Model':        'Optuna LightGBM (Bal)',
    'Accuracy':     accuracy_score(y_test, lgb_optuna_pred_bal),
    'Balanced Acc': balanced_accuracy_score(y_test, lgb_optuna_pred_bal),
    'Sensitivity':  recall_score(y_test, lgb_optuna_pred_bal, pos_label=1),
    'Specificity':  recall_score(y_test, lgb_optuna_pred_bal, pos_label=0),
    'AUC':          roc_auc_score(y_test, best_lgb_optuna_bal.predict_proba(X_test_ready)[:, 1]),
}]).round(4)
print(lgb_optuna_bal_results.to_string(index=False))


[I 2026-04-29 17:29:25,161] A new study created in memory with name: no-name-80486fa9-a503-4152-a8cd-223c924f843d
[I 2026-04-29 17:29:27,102] Trial 0 finished with value: 0.7312393571914635 and parameters: {'n_estimators': 250, 'learning_rate': 0.08927180304353628, 'num_leaves': 102, 'max_depth': 14, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.0001707396743152812, 'reg_lambda': 0.29154431891537524, 'min_child_samples': 34}. Best is trial 0 with value: 0.7312393571914635.
[I 2026-04-29 17:29:30,573] Trial 1 finished with value: 0.7546668976687052 and parameters: {'n_estimators': 383, 'learning_rate': 0.010485387725194618, 'num_leaves': 125, 'max_depth': 18, 'subsample': 0.7637017332034828, 'colsample_bytree': 0.7545474901621302, 'reg_alpha': 0.0005415244119402539, 'reg_lambda': 0.0016480446427978971, 'min_child_samples': 31}. Best is trial 1 with value: 0.7546668976687052.
[I 2026-04-29 17:29:32,174] Trial 2 finished with value: 0.756538896278

Best LGB Optuna (balanced) params: {'n_estimators': 254, 'learning_rate': 0.025345662505675864, 'num_leaves': 34, 'max_depth': 11, 'subsample': 0.9388841740177296, 'colsample_bytree': 0.8412749859649511, 'reg_alpha': 0.011636270301075117, 'reg_lambda': 0.0003643678722194405, 'min_child_samples': 50}
Best LGB Optuna balanced accuracy: 0.7589
                Model  Accuracy  Balanced Acc  Sensitivity  Specificity    AUC
Optuna LightGBM (Bal)    0.8408        0.7358       0.6002       0.8714 0.7818


### 4.4 Optuna Search: XGBoost

Run an Optuna search for XGBoost and evaluate the selected pipeline on the test set.


In [ ]:
# Optuna for XGBoost — Section 2 (balanced_accuracy, SMOTE inside objective)

def xgb_objective_bal(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    pipe = Pipeline([
        ('model', xgb.XGBClassifier(
            **params,
            eval_metric='logloss', tree_method='hist',
            device=xgb_device, random_state=42, n_jobs=1, scale_pos_weight=scale_pos
        )),
    ])
    scores = cross_val_score(
        pipe, X_train_ready, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='balanced_accuracy', n_jobs=1
    )
    return scores.mean()

xgb_study_bal = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
xgb_study_bal.optimize(xgb_objective_bal, n_trials=75)

print(f"Best XGB Optuna (balanced) params: {xgb_study_bal.best_params}")
print(f"Best XGB Optuna balanced accuracy: {xgb_study_bal.best_value:.4f}")

best_xgb_optuna_bal = Pipeline([
    ('model', xgb.XGBClassifier(
        **xgb_study_bal.best_params,
        eval_metric='logloss', tree_method='hist',
        device=xgb_device, random_state=42, n_jobs=-1, scale_pos_weight=scale_pos
    )),
])
best_xgb_optuna_bal.fit(X_train_ready, y_train)
xgb_optuna_pred_bal = best_xgb_optuna_bal.predict(X_test_ready)

xgb_optuna_bal_results = pd.DataFrame([{
    'Model':        'Optuna XGBoost (Bal)',
    'Accuracy':     accuracy_score(y_test, xgb_optuna_pred_bal),
    'Balanced Acc': balanced_accuracy_score(y_test, xgb_optuna_pred_bal),
    'Sensitivity':  recall_score(y_test, xgb_optuna_pred_bal, pos_label=1),
    'Specificity':  recall_score(y_test, xgb_optuna_pred_bal, pos_label=0),
    'AUC':          roc_auc_score(y_test, best_xgb_optuna_bal.predict_proba(X_test_ready)[:, 1]),
}]).round(4)
print(xgb_optuna_bal_results.to_string(index=False))


[I 2026-04-29 17:34:31,340] A new study created in memory with name: no-name-6606b0ef-c179-41a8-998e-fd2a20d878c5
[I 2026-04-29 17:34:33,471] Trial 0 finished with value: 0.7421595351401182 and parameters: {'n_estimators': 250, 'learning_rate': 0.08927180304353628, 'max_depth': 7, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'reg_alpha': 0.0004207053950287938, 'reg_lambda': 0.0001707396743152812, 'min_child_weight': 9}. Best is trial 0 with value: 0.7421595351401182.
[I 2026-04-29 17:34:35,212] Trial 1 finished with value: 0.7529049803913229 and parameters: {'n_estimators': 341, 'learning_rate': 0.051059032093947576, 'max_depth': 3, 'subsample': 0.9909729556485983, 'colsample_bytree': 0.9497327922401265, 'reg_alpha': 0.0007068974950624604, 'reg_lambda': 0.000533703276260396, 'min_child_weight': 2}. Best is trial 1 with value: 0.7529049803913229.
[I 2026-04-29 17:34:36,792] Trial 2 finished with value: 0.7538553193953537 and parameters: {'n_estimators': 222, 

Best XGB Optuna (balanced) params: {'n_estimators': 495, 'learning_rate': 0.012524877695681322, 'max_depth': 6, 'subsample': 0.727684014854839, 'colsample_bytree': 0.7632731797007789, 'reg_alpha': 0.001914272447543007, 'reg_lambda': 0.9972100672063969, 'min_child_weight': 8}
Best XGB Optuna balanced accuracy: 0.7570
               Model  Accuracy  Balanced Acc  Sensitivity  Specificity    AUC
Optuna XGBoost (Bal)    0.8388        0.7342       0.5991       0.8692 0.7832


### 4.5 Final Classical Model Evaluation

Assemble tuned and Optuna-selected model results for a final classical-model comparison.


In [ ]:
# Section 2 Final Test Set Evaluation + cross-section comparison

def eval_row_bal(name, model, pred):
    return {
        'Model':        name,
        'Accuracy':     accuracy_score(y_test, pred),
        'Balanced Acc': balanced_accuracy_score(y_test, pred),
        'Sensitivity':  recall_score(y_test, pred, pos_label=1),
        'Specificity':  recall_score(y_test, pred, pos_label=0),
        'AUC':          roc_auc_score(y_test, model.predict_proba(X_test_ready)[:, 1]),
    }

final_results_bal = pd.DataFrame([
    eval_row_bal('Tuned LightGBM (Bal)',      best_lgb_bal,         lgb_pred_bal),
    eval_row_bal('Tuned Random Forest (Bal)',  best_rf_bal,          rf_pred_bal),
    eval_row_bal('Tuned XGBoost (Bal)',        best_xgb_bal,         xgb_pred_bal),
    eval_row_bal('Optuna LightGBM (Bal)',      best_lgb_optuna_bal,  lgb_optuna_pred_bal),
    eval_row_bal('Optuna XGBoost (Bal)',       best_xgb_optuna_bal,  xgb_optuna_pred_bal),
]).round(4).sort_values('Balanced Acc', ascending=False)

print("\n=== SECTION 2 FINAL TEST SET RESULTS (SMOTE/Native Imbalance Handling + Balanced Accuracy) ===")
print(final_results_bal.to_string(index=False))

# ── Section 1 vs Section 2: best model comparison (skipped if Section 1 not run) ──
'''try:
    best_s1 = final_results.sort_values('Accuracy', ascending=False).iloc[0].to_dict()
    best_s2 = final_results_bal.sort_values('Balanced Acc', ascending=False).iloc[0].to_dict()

    comparison = pd.DataFrame([
        {**best_s1, 'Section': 'Section 1 (Accuracy-First, No Balancing)'},
        {**best_s2, 'Section': 'Section 2 (SMOTE + Balanced Accuracy)'},
    ]).set_index('Section')[['Model', 'Accuracy', 'Balanced Acc', 'Sensitivity', 'Specificity', 'AUC']].round(4)

    print("\n=== SECTION 1 vs SECTION 2: BEST MODEL COMPARISON ===")
    print(comparison.to_string())
except NameError:
    print("\n(Section 1 results not available — skipping cross-section comparison.)")'''



=== SECTION 2 FINAL TEST SET RESULTS (SMOTE/Native Imbalance Handling + Balanced Accuracy) ===
                    Model  Accuracy  Balanced Acc  Sensitivity  Specificity    AUC
    Optuna LightGBM (Bal)    0.8408        0.7358       0.6002       0.8714 0.7818
     Optuna XGBoost (Bal)    0.8388        0.7342       0.5991       0.8692 0.7832
     Tuned LightGBM (Bal)    0.8384        0.7321       0.5948       0.8693 0.7818
Tuned Random Forest (Bal)    0.8603        0.7275       0.5560       0.8989 0.7764
      Tuned XGBoost (Bal)    0.8017        0.7251       0.6261       0.8241 0.7726


'try:\n    best_s1 = final_results.sort_values(\'Accuracy\', ascending=False).iloc[0].to_dict()\n    best_s2 = final_results_bal.sort_values(\'Balanced Acc\', ascending=False).iloc[0].to_dict()\n\n    comparison = pd.DataFrame([\n        {**best_s1, \'Section\': \'Section 1 (Accuracy-First, No Balancing)\'},\n        {**best_s2, \'Section\': \'Section 2 (SMOTE + Balanced Accuracy)\'},\n    ]).set_index(\'Section\')[[\'Model\', \'Accuracy\', \'Balanced Acc\', \'Sensitivity\', \'Specificity\', \'AUC\']].round(4)\n\n    print("\n=== SECTION 1 vs SECTION 2: BEST MODEL COMPARISON ===")\n    print(comparison.to_string())\nexcept NameError:\n    print("\n(Section 1 results not available — skipping cross-section comparison.)")'

## 5. Transformer-Based Model for Bank Marketing Prediction

Train a tabular transformer that embeds categorical fields, projects numeric fields into tokens, adds a `[CLS]` token, and predicts whether a client subscribes to the campaign offer.


### 5.1 Runtime Setup

Run this section in a GPU-enabled Colab session when possible. The notebook will fall back to CPU, but the transformer training and optional XGBoost baseline are much slower without GPU acceleration.


In [1]:
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

SEED = 580
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


### 5.2 Reload Data for the Transformer Workflow

Reload the raw CSV files so the transformer track starts from the original labels and columns rather than the earlier one-hot encoded classical-model matrices.


In [2]:
train_df = pd.read_csv("./training_data.csv")
test_df = pd.read_csv("./testing_data.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df["y"].value_counts(normalize=True).rename("class_rate"))

Train shape: (32951, 21)
Test shape: (8237, 21)
y
no     0.887348
yes    0.112652
Name: class_rate, dtype: float64


### 5.3 Evaluation Metrics and Threshold Selection

Define reusable metrics and validation-driven threshold selection rules for comparing default, accuracy-focused, harmonic, and business-score operating points.


In [3]:
def safe_div(numerator, denominator):
    return np.nan if denominator == 0 else numerator / denominator


def metric_row(y_true, prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob)
    pred = (prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    accuracy = safe_div(tp + tn, len(y_true))
    sensitivity = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    precision = safe_div(tp, tp + fp)
    f1 = safe_div(2 * precision * sensitivity, precision + sensitivity)
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    if min(accuracy, sensitivity, specificity) <= 0:
        harmonic = 0.0
    else:
        harmonic = 3.0 / ((1.0 / accuracy) + (1.0 / sensitivity) + (1.0 / specificity))
    business_score = 0.4 * accuracy + 0.4 * sensitivity + 0.2 * specificity
    return {
        "threshold": threshold,
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy,
        "harmonic_acc_sens_spec": harmonic,
        "business_score": business_score,
        "precision": precision,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def threshold_table(y_true, prob):
    rows = [metric_row(y_true, prob, th) for th in np.r_[np.arange(0.01, 0.991, 0.001), 0.5]]
    out = pd.DataFrame(rows).drop_duplicates("threshold")
    out["auc"] = roc_auc_score(y_true, prob)
    return out


def pick_threshold(y_true, prob, objective="harmonic_acc_sens_spec"):
    tab = threshold_table(y_true, prob)
    return tab.sort_values(
        [objective, "balanced_accuracy", "accuracy"],
        ascending=[False, False, False],
    ).iloc[0]


def evaluate_points(model_name, y_true, prob, threshold_source_y, threshold_source_prob):
    rows = []
    for name, th in [
        ("default_0.50", 0.5),
        ("accuracy_threshold", pick_threshold(threshold_source_y, threshold_source_prob, "accuracy")["threshold"]),
        ("harmonic_threshold", pick_threshold(threshold_source_y, threshold_source_prob, "harmonic_acc_sens_spec")["threshold"]),
        ("business_threshold", pick_threshold(threshold_source_y, threshold_source_prob, "business_score")["threshold"]),
    ]:
        row = metric_row(y_true, prob, float(th))
        row["model"] = model_name
        row["operating_point"] = name
        row["auc"] = roc_auc_score(y_true, prob)
        rows.append(row)
    return pd.DataFrame(rows)

### 5.4 Feature Engineering and Tabular Preprocessing

Create additional campaign, contact-history, economic-regime, and interaction features; fit categorical maps and numeric scaling on the transformer training split only.


In [4]:
def education_order(values):
    mapping = {
        "illiterate": 0,
        "basic.4y": 1,
        "basic.6y": 2,
        "basic.9y": 3,
        "high.school": 4,
        "professional.course": 5,
        "university.degree": 6,
        "unknown": np.nan,
    }
    out = pd.Series(values).map(mapping).astype(float)
    return out.fillna(out.median()).to_numpy()


def add_features(df, campaign_cap=None):
    out = df.copy()
    if "y" in out.columns:
        out = out.drop(columns=["y"])
    if "duration" in out.columns:
        out = out.drop(columns=["duration"])

    if campaign_cap is None:
        campaign_cap = float(out["campaign"].quantile(0.99))

    out["campaign_capped"] = out["campaign"].clip(upper=campaign_cap)
    out["campaign_log"] = np.log1p(out["campaign"])
    out["campaign_group"] = pd.cut(
        out["campaign_capped"],
        bins=[-np.inf, 1, 2, 4, 8, np.inf],
        labels=["one", "two", "three_four", "five_eight", "nine_plus"],
    ).astype(str)
    out["age_bin"] = pd.cut(
        out["age"],
        bins=[-np.inf, 25, 35, 45, 55, 65, np.inf],
        labels=["under_25", "25_35", "35_45", "45_55", "55_65", "over_65"],
    ).astype(str)
    out["education_ord"] = education_order(out["education"])
    out["pdays_missing"] = (out["pdays"] == 999).astype(int)
    out["pdays_clean"] = np.where(out["pdays"] == 999, 0, out["pdays"])
    out["pdays_recent"] = (out["pdays"] <= 7).astype(int)
    out["contacted_before"] = ((out["pdays"] != 999) | (out["previous"] > 0)).astype(int)
    out["previous_success"] = (out["poutcome"] == "success").astype(int)
    out["previous_failure"] = (out["poutcome"] == "failure").astype(int)
    out["previous_group"] = pd.cut(
        out["previous"],
        bins=[-np.inf, 0, 1, 2, np.inf],
        labels=["none", "one", "two", "three_plus"],
    ).astype(str)
    out["emp_regime"] = pd.cut(
        out["emp.var.rate"],
        bins=[-np.inf, -2, 0, 1, np.inf],
        labels=["recession", "weak", "neutral", "expansion"],
    ).astype(str)
    out["euribor_regime"] = pd.cut(
        out["euribor3m"],
        bins=[-np.inf, 1, 2, 4, np.inf],
        labels=["very_low", "low", "medium", "high"],
    ).astype(str)
    out["confidence_regime"] = pd.cut(
        out["cons.conf.idx"],
        bins=[-np.inf, -45, -40, -35, np.inf],
        labels=["very_low", "low", "medium", "high"],
    ).astype(str)
    out["employment_regime"] = pd.cut(
        out["nr.employed"],
        bins=[-np.inf, 5050, 5150, 5200, np.inf],
        labels=["low", "medium", "high", "very_high"],
    ).astype(str)
    out["contact_month"] = out["contact"].astype(str) + "_" + out["month"].astype(str)
    out["poutcome_previous"] = out["poutcome"].astype(str) + "_" + out["previous_group"].astype(str)
    pdays_status = pd.Series(
        np.where(out["pdays_missing"] == 1, "not_contacted", "contacted"),
        index=out.index,
    )
    out["pdays_previous"] = pdays_status.astype(str) + "_" + out["previous_group"].astype(str)
    out["job_education"] = out["job"].astype(str) + "_" + out["education"].astype(str)
    out["contact_poutcome"] = out["contact"].astype(str) + "_" + out["poutcome"].astype(str)
    out["month_emp_regime"] = out["month"].astype(str) + "_" + out["emp_regime"].astype(str)
    return out, campaign_cap


class TabPreprocessor:
    def __init__(self, min_count=30):
        self.min_count = min_count

    def fit(self, df):
        frame, self.campaign_cap = add_features(df)
        self.cat_cols = frame.select_dtypes(include=["object", "category"]).columns.tolist()
        self.num_cols = [c for c in frame.columns if c not in self.cat_cols]

        self.cat_maps = {}
        self.cardinalities = []
        for col in self.cat_cols:
            values = frame[col].fillna("__missing__").astype(str)
            counts = values.value_counts()
            keep = counts[counts >= self.min_count].index.tolist()
            if not keep:
                keep = counts.index[:1].tolist()
            levels = ["__unknown__"] + sorted(keep)
            self.cat_maps[col] = {level: i for i, level in enumerate(levels)}
            self.cardinalities.append(len(levels))

        num = frame[self.num_cols].astype(float)
        self.num_mean = num.mean()
        self.num_std = num.std().replace(0, 1).fillna(1)
        return self

    def transform(self, df):
        frame, _ = add_features(df, self.campaign_cap)
        cat_arrays = []
        for col in self.cat_cols:
            mapping = self.cat_maps[col]
            values = frame[col].fillna("__missing__").astype(str)
            cat_arrays.append(values.map(mapping).fillna(0).astype("int64").to_numpy())
        x_cat = np.stack(cat_arrays, axis=1).astype("int64")
        x_num = ((frame[self.num_cols].astype(float) - self.num_mean) / self.num_std).fillna(0).to_numpy("float32")
        return x_cat, x_num


y = (train_df["y"] == "yes").astype("int64").to_numpy()
y_test = (test_df["y"] == "yes").astype("int64").to_numpy()

train_idx, valid_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

pre = TabPreprocessor(min_count=30).fit(train_df.iloc[train_idx])
train_cat, train_num = pre.transform(train_df.iloc[train_idx])
valid_cat, valid_num = pre.transform(train_df.iloc[valid_idx])
test_cat, test_num = pre.transform(test_df)
y_train = y[train_idx]
y_valid = y[valid_idx]

print("Categorical columns:", len(pre.cat_cols))
print("Numerical columns:", len(pre.num_cols))
print("Categorical cardinalities:", pre.cardinalities)

Categorical columns: 23
Numerical columns: 18
Categorical cardinalities: [13, 5, 8, 3, 4, 4, 3, 11, 6, 4, 6, 7, 5, 4, 4, 5, 5, 20, 8, 8, 73, 7, 21]


### 5.5 Dataset and Model Implementation

Wrap the preprocessed tabular arrays in a PyTorch dataset and define the FT-Transformer architecture used for binary classification.


In [5]:
class BankDataset(Dataset):
    def __init__(self, x_cat, x_num, y=None):
        self.x_cat = torch.as_tensor(x_cat, dtype=torch.long)
        self.x_num = torch.as_tensor(x_num, dtype=torch.float32)
        self.y = None if y is None else torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x_cat)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x_cat[idx], self.x_num[idx]
        return self.x_cat[idx], self.x_num[idx], self.y[idx]


class FTTransformer(nn.Module):
    def __init__(self, cat_cardinalities, n_num, d_token=48, n_heads=4, n_layers=3, dropout=0.15):
        super().__init__()
        if d_token % n_heads != 0:
            raise ValueError("d_token must be divisible by n_heads")
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(cardinality, d_token) for cardinality in cat_cardinalities
        ])
        self.num_projections = nn.ModuleList([
            nn.Sequential(nn.Linear(1, d_token), nn.LayerNorm(d_token)) for _ in range(n_num)
        ])
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token, 1),
        )
        nn.init.normal_(self.cls, std=0.02)

    def forward(self, x_cat, x_num):
        tokens = []
        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat[:, i]))
        for i, proj in enumerate(self.num_projections):
            tokens.append(proj(x_num[:, i : i + 1]))
        x = torch.stack(tokens, dim=1)
        cls = self.cls.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.encoder(x)
        return self.head(x[:, 0]).squeeze(1)

### 5.6 Training and Model Selection

Train several transformer configurations, track validation AUC for early stopping, and evaluate each selected model on the held-out test set.


In [6]:
def predict_proba(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                x_cat, x_num, _ = batch
            else:
                x_cat, x_num = batch
            x_cat = x_cat.to(DEVICE)
            x_num = x_num.to(DEVICE)
            logits = model(x_cat, x_num)
            probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)


def train_one(config):
    batch_size = config.get("batch_size", 512)
    train_loader = DataLoader(
        BankDataset(train_cat, train_num, y_train),
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda"),
    )
    valid_loader = DataLoader(
        BankDataset(valid_cat, valid_num, y_valid),
        batch_size=2048,
        shuffle=False,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda"),
    )
    model = FTTransformer(
        pre.cardinalities,
        train_num.shape[1],
        d_token=config["d_token"],
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        dropout=config["dropout"],
    ).to(DEVICE)

    pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32, device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_auc = -np.inf
    best_state = None
    stale_epochs = 0
    start = time.time()

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        running_loss = 0.0
        for x_cat, x_num, target in train_loader:
            x_cat = x_cat.to(DEVICE)
            x_num = x_num.to(DEVICE)
            target = target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits = model(x_cat, x_num)
                loss = loss_fn(logits, target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * len(target)

        valid_prob = predict_proba(model, valid_loader)
        valid_auc = roc_auc_score(y_valid, valid_prob)
        if valid_auc > best_auc:
            best_auc = valid_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1

        print(
            f"epoch {epoch:03d} | loss {running_loss / len(y_train):.4f} | valid_auc {valid_auc:.4f} | best {best_auc:.4f}"
        )
        if stale_epochs >= config["patience"]:
            break

    model.load_state_dict(best_state)
    elapsed = time.time() - start
    return model, best_auc, elapsed


CONFIGS = [
    {"name": "ft_small", "d_token": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.15, "lr": 3e-4, "weight_decay": 1e-4, "batch_size": 512, "epochs": 50, "patience": 8},
    {"name": "ft_medium", "d_token": 48, "n_heads": 4, "n_layers": 3, "dropout": 0.20, "lr": 2e-4, "weight_decay": 3e-4, "batch_size": 512, "epochs": 60, "patience": 10},
    {"name": "ft_wide", "d_token": 64, "n_heads": 8, "n_layers": 3, "dropout": 0.15, "lr": 2e-4, "weight_decay": 1e-4, "batch_size": 512, "epochs": 60, "patience": 10},
]

results = []
best = None
for config in CONFIGS:
    print("\nTraining", config["name"])
    model, best_auc, elapsed = train_one(config)
    valid_loader = DataLoader(BankDataset(valid_cat, valid_num, y_valid), batch_size=2048, shuffle=False)
    test_loader = DataLoader(BankDataset(test_cat, test_num, y_test), batch_size=2048, shuffle=False)
    valid_prob = predict_proba(model, valid_loader)
    test_prob = predict_proba(model, test_loader)
    metrics = evaluate_points(config["name"], y_test, test_prob, y_valid, valid_prob)
    metrics["valid_auc"] = best_auc
    metrics["seconds"] = elapsed
    results.append(metrics)
    if best is None or best_auc > best["valid_auc"]:
        best = {"config": config, "model": model, "valid_auc": best_auc, "valid_prob": valid_prob, "test_prob": test_prob}

transformer_metrics = pd.concat(results, ignore_index=True)
transformer_metrics.sort_values("harmonic_acc_sens_spec", ascending=False)


Training ft_small


/tmp/ipykernel_3083/4087044970.py:37: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
/tmp/ipykernel_3083/2317673663.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
/tmp/ipykernel_3083/2317673663.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


epoch 001 | loss 1.1166 | valid_auc 0.7735 | best 0.7735


/tmp/ipykernel_3083/2317673663.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


epoch 002 | loss 1.0179 | valid_auc 0.7861 | best 0.7861


...

/tmp/ipykernel_3083/2317673663.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


epoch 022 | loss 0.9215 | valid_auc 0.7946 | best 0.7993


/tmp/ipykernel_3083/2317673663.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


epoch 023 | loss 0.9182 | valid_auc 0.7918 | best 0.7993


,threshold,accuracy,sensitivity,specificity,balanced_accuracy,harmonic_acc_sens_spec,business_score,precision,f1,tp,tn,fp,fn,model,operating_point,auc,valid_auc,seconds
0,0.500,0.844361,0.581897,0.877685,0.729791,0.742169,0.746040,0.376569,0.457240,540,6415,894,388,ft_small,default_0.50,0.770383,0.797234,12.004963
10,0.444,0.835620,0.589440,0.866876,0.728158,0.741325,0.743399,0.359868,0.446895,547,6336,973,381,ft_wide,harmonic_threshold,0.776117,0.799333,12.042159
7,0.540,0.840112,0.581897,0.872896,0.727396,0.739928,0.743383,0.367597,0.450563,540,6380,929,388,ft_medium,business_threshold,0.774430,0.797337,22.570597
3,0.527,0.849703,0.571121,0.885073,0.728097,0.739339,0.745344,0.386861,0.461271,530,6469,840,398,ft_small,business_threshold,0.770383,0.797234,12.004963
2,0.428,0.815103,0.605603,0.841702,0.723653,0.737798,0.736623,0.326934,0.424632,562,6152,1157,366,ft_small,harmonic_threshold,0.770383,0.797234,12.004963
11,0.470,0.842297,0.575431,0.876180,0.725806,0.737758,0.742327,0.371091,0.451204,534,6404,905,394,ft_wide,business_threshold,0.776117,0.799333,12.042159
6,0.483,0.808790,0.612069,0.833767,0.722918,0.737174,0.735097,0.318564,0.419034,568,6094,1215,360,ft_medium,harmonic_threshold,0.774430,0.797337,22.570597
4,0.500,0.822994,0.594828,0.851963,0.723395,0.737102,0.737521,0.337821,0.430913,552,6227,1082,376,ft_medium,default_0.50,0.774430,0.797337,22.570597
8,0.500,0.848610,0.567888,0.884252,0.726070,0.737063,0.743450,0.383831,0.458062,527,6463,846,401,ft_wide,default_0.50,0.776117,0.799333,12.042159
9,0.906,0.902756,0.229526,0.988234,0.608880,0.463218,0.650559,0.712375,0.347188,213,7223,86,715,ft_wide,accuracy_threshold,0.776117,0.799333,12.042159


### 5.7 Transformer Results

Display operating-point metrics for the transformer configurations and identify the best validation-AUC configuration.


In [7]:
display_cols = [
    "model",
    "operating_point",
    "threshold",
    "accuracy",
    "sensitivity",
    "specificity",
    "balanced_accuracy",
    "harmonic_acc_sens_spec",
    "business_score",
    "auc",
    "valid_auc",
    "tp",
    "tn",
    "fp",
    "fn",
]

summary = transformer_metrics[display_cols].sort_values(
    ["harmonic_acc_sens_spec", "business_score"],
    ascending=False,
)
display(summary)
print("Best config by validation AUC:", best["config"])

,model,operating_point,threshold,accuracy,sensitivity,specificity,balanced_accuracy,harmonic_acc_sens_spec,business_score,auc,valid_auc,tp,tn,fp,fn
0,ft_small,default_0.50,0.500,0.844361,0.581897,0.877685,0.729791,0.742169,0.746040,0.770383,0.797234,540,6415,894,388
10,ft_wide,harmonic_threshold,0.444,0.835620,0.589440,0.866876,0.728158,0.741325,0.743399,0.776117,0.799333,547,6336,973,381
7,ft_medium,business_threshold,0.540,0.840112,0.581897,0.872896,0.727396,0.739928,0.743383,0.774430,0.797337,540,6380,929,388
3,ft_small,business_threshold,0.527,0.849703,0.571121,0.885073,0.728097,0.739339,0.745344,0.770383,0.797234,530,6469,840,398
2,ft_small,harmonic_threshold,0.428,0.815103,0.605603,0.841702,0.723653,0.737798,0.736623,0.770383,0.797234,562,6152,1157,366
11,ft_wide,business_threshold,0.470,0.842297,0.575431,0.876180,0.725806,0.737758,0.742327,0.776117,0.799333,534,6404,905,394
6,ft_medium,harmonic_threshold,0.483,0.808790,0.612069,0.833767,0.722918,0.737174,0.735097,0.774430,0.797337,568,6094,1215,360
4,ft_medium,default_0.50,0.500,0.822994,0.594828,0.851963,0.723395,0.737102,0.737521,0.774430,0.797337,552,6227,1082,376
8,ft_wide,default_0.50,0.500,0.848610,0.567888,0.884252,0.726070,0.737063,0.743450,0.776117,0.799333,527,6463,846,401
9,ft_wide,accuracy_threshold,0.906,0.902756,0.229526,0.988234,0.608880,0.463218,0.650559,0.776117,0.799333,213,7223,86,715


Best config by validation AUC: {'name': 'ft_wide', 'd_token': 64, 'n_heads': 8, 'n_layers': 3, 'dropout': 0.15, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 512, 'epochs': 60, 'patience': 10}


### 5.8 Optional XGBoost Baseline

Train a GPU-aware XGBoost baseline using the same feature-engineering helper functions for an additional comparison point against the transformer models.


In [8]:
# Optional GPU XGBoost baseline. This is useful because tree boosting is usually very strong
# on small/medium tabular data like this dataset.
try:
    import xgboost as xgb
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    feature_train, cap = add_features(train_df.iloc[train_idx])
    feature_valid, _ = add_features(train_df.iloc[valid_idx], cap)
    feature_test, _ = add_features(test_df, cap)

    cat_cols = feature_train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in feature_train.columns if c not in cat_cols]

    pre_xgb = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), cat_cols),
            ("num", StandardScaler(), num_cols),
        ]
    )

    xgb_kwargs = dict(
        n_estimators=700,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=1,
        subsample=0.90,
        colsample_bytree=0.70,
        reg_lambda=3,
        reg_alpha=1,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=math.sqrt((y_train == 0).sum() / (y_train == 1).sum()),
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist",
    )
    if DEVICE.type == "cuda":
        xgb_kwargs["device"] = "cuda"

    xgb_model = Pipeline([
        ("prep", pre_xgb),
        ("model", xgb.XGBClassifier(**xgb_kwargs)),
    ])

    try:
        xgb_model.fit(feature_train, y_train)
    except Exception as first_error:
        print("Retrying XGBoost without device='cuda' because:", first_error)
        xgb_kwargs.pop("device", None)
        if DEVICE.type == "cuda":
            xgb_kwargs["tree_method"] = "gpu_hist"
        xgb_model = Pipeline([
            ("prep", pre_xgb),
            ("model", xgb.XGBClassifier(**xgb_kwargs)),
        ])
        xgb_model.fit(feature_train, y_train)

    valid_prob_xgb = xgb_model.predict_proba(feature_valid)[:, 1]
    test_prob_xgb = xgb_model.predict_proba(feature_test)[:, 1]
    xgb_metrics = evaluate_points("xgboost_gpu_baseline", y_test, test_prob_xgb, y_valid, valid_prob_xgb)
    combined = pd.concat([transformer_metrics, xgb_metrics], ignore_index=True)
    display(combined[display_cols[:-1] + ["tp", "tn", "fp", "fn"]].sort_values("harmonic_acc_sens_spec", ascending=False))
except Exception as exc:
    print("Skipped optional XGBoost baseline:", repr(exc))

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [00:39:47] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,model,operating_point,threshold,accuracy,sensitivity,specificity,balanced_accuracy,harmonic_acc_sens_spec,business_score,auc,valid_auc,tp,tn,fp,tp,tn,fp,fn
0,ft_small,default_0.50,0.500,0.844361,0.581897,0.877685,0.729791,0.742169,0.746040,0.770383,0.797234,540,6415,894,540,6415,894,388
10,ft_wide,harmonic_threshold,0.444,0.835620,0.589440,0.866876,0.728158,0.741325,0.743399,0.776117,0.799333,547,6336,973,547,6336,973,381
7,ft_medium,business_threshold,0.540,0.840112,0.581897,0.872896,0.727396,0.739928,0.743383,0.774430,0.797337,540,6380,929,540,6380,929,388
3,ft_small,business_threshold,0.527,0.849703,0.571121,0.885073,0.728097,0.739339,0.745344,0.770383,0.797234,530,6469,840,530,6469,840,398
2,ft_small,harmonic_threshold,0.428,0.815103,0.605603,0.841702,0.723653,0.737798,0.736623,0.770383,0.797234,562,6152,1157,562,6152,1157,366
11,ft_wide,business_threshold,0.470,0.842297,0.575431,0.876180,0.725806,0.737758,0.742327,0.776117,0.799333,534,6404,905,534,6404,905,394
15,xgboost_gpu_baseline,business_threshold,0.298,0.842176,0.575431,0.876043,0.725737,0.737694,0.742251,0.774333,NaN,534,6403,906,534,6403,906,394
14,xgboost_gpu_baseline,harmonic_threshold,0.298,0.842176,0.575431,0.876043,0.725737,0.737694,0.742251,0.774333,NaN,534,6403,906,534,6403,906,394
6,ft_medium,harmonic_threshold,0.483,0.808790,0.612069,0.833767,0.722918,0.737174,0.735097,0.774430,0.797337,568,6094,1215,568,6094,1215,360
4,ft_medium,default_0.50,0.500,0.822994,0.594828,0.851963,0.723395,0.737102,0.737521,0.774430,0.797337,552,6227,1082,552,6227,1082,376


### 5.9 Save Artifacts

Persist the selected transformer state, preprocessing metadata, metrics, and prediction probabilities to the Colab output directory.


In [9]:
output_dir = Path("/content/transformer_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

best_model_path = output_dir / "ft_transformer_bank_model.pt"
torch.save(
    {
        "state_dict": best["model"].state_dict(),
        "config": best["config"],
        "cat_cols": pre.cat_cols,
        "num_cols": pre.num_cols,
        "cat_maps": pre.cat_maps,
        "cardinalities": pre.cardinalities,
        "campaign_cap": pre.campaign_cap,
        "num_mean": pre.num_mean.to_dict(),
        "num_std": pre.num_std.to_dict(),
    },
    best_model_path,
)

transformer_metrics.to_csv(output_dir / "transformer_test_metrics.csv", index=False)
pd.DataFrame(
    {
        "row_id": np.arange(len(y_test)) + 1,
        "actual": np.where(y_test == 1, "yes", "no"),
        "probability_yes": best["test_prob"],
    }
).to_csv(output_dir / "transformer_test_predictions.csv", index=False)

print("Saved:")
print(best_model_path)
print(output_dir / "transformer_test_metrics.csv")
print(output_dir / "transformer_test_predictions.csv")

Saved:
/content/transformer_outputs/ft_transformer_bank_model.pt
/content/transformer_outputs/transformer_test_metrics.csv
/content/transformer_outputs/transformer_test_predictions.csv
